In [1]:
!pip install -U transformers
import transformers
print(transformers.__version__)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 82.8 MB/s eta 0:00:00:00:01:01
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
5.3.0


In [2]:
import transformers
print(transformers.__version__)


5.3.0


In [3]:
# !pip install -U torch transformers datasets accelerate sentencepiece pandas numpy tqdm

import json
import time
import math
import re
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from tqdm.auto import tqdm

In [4]:
from huggingface_hub import hf_hub_download
import zipfile


file_path = hf_hub_download(
    repo_id="zai-org/LongBench",
    filename="data.zip",
    repo_type="dataset"
)


zip_path = file_path   # path returned by hf_hub_download
extract_dir = "./longbench_data"

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(extract_dir)


data.zip:   0%|          | 0.00/114M [00:00<?, ?B/s]

In [5]:
from datasets import load_dataset, Dataset

ds = load_dataset("simonjegou/ruler", "16384" , split="test")
# print(ds[0])
# ds.to_json("ruler_16384.jsonl")
import os

os.makedirs("ruler_data", exist_ok=True)

for task_name in set(ds["task"]):
    subset = ds.filter(lambda ex: ex["task"] == task_name)
    out_path = f"ruler_data/{task_name}.jsonl"
    subset.to_json(out_path)
    print(f"Saved {task_name} → {out_path}")



README.md: 0.00B [00:00, ?B/s]

16384/test-00000-of-00001.parquet:   0%|          | 0.00/189M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/6500 [00:00<?, ? examples/s]

Filter:   0%|          | 0/6500 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

...

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved vt → ruler_data/vt.jsonl


Filter:   0%|          | 0/6500 [00:00<?, ? examples/s]

Creating json from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Saved qa_1 → ruler_data/qa_1.jsonl


In [6]:
# !pip install "transformers[serving] @ git+https://github.com/huggingface/transformers.git@main"

from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen3.5-27B"#"Qwen/Qwen2.5-7B-Instruct" #"Qwen/Qwen2.5-32B" #
DTYPE = torch.float16 if torch.cuda.is_available() else torch.float32
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=DTYPE,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)
if DEVICE == "cpu":
    model = model.to(DEVICE)

# Critical for ThinKV/attention-based scoring:
# sdpa/flash kernels often do not return attention weights.
try:
    if hasattr(model, "set_attn_implementation"):
        model.set_attn_implementation("eager")
    elif hasattr(model.config, "_attn_implementation"):
        model.config._attn_implementation = "eager"
except Exception as e:
    print("[WARN] Could not force eager attention:", e)

model.eval()
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token

print("loaded", MODEL_NAME)
print("attn_impl:", getattr(model.config, "_attn_implementation", "unknown"))


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/12.8M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


...

The fast path is not available because one of the required library is not installed. Falling back to torch implementation. To install follow https://github.com/fla-org/flash-linear-attention#installation and https://github.com/Dao-AILab/causal-conv1d


Loading weights:   0%|          | 0/851 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/244 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the cpu.


loaded Qwen/Qwen3.5-27B
attn_impl: eager


In [7]:
def _pair_layers(n_layers: int):
    pairs = []
    i = 0
    while i < n_layers:
        if i + 1 < n_layers:
            pairs.append((i, i + 1))
            i += 2
        else:
            pairs.append((i, None))
            i += 1
    return pairs


def _norm_dir(x, eps=1e-6):
    mag = torch.norm(x, dim=-1, keepdim=True)
    return x / (mag + eps), mag


def _bytes_tensor(x: torch.Tensor) -> int:
    return x.numel() * x.element_size()


def _stack_layers(layer_list):
    return tuple((k, v) for k, v in layer_list)

def cache_to_legacy(past_key_values):
    """Convert HF Cache object to legacy tuple[(k,v), ...] when needed."""
    if past_key_values is None:
        return None

    if isinstance(past_key_values, (tuple, list)):
        return tuple(past_key_values)

    for m in ["to_legacy_cache", "to_legacy"]:
        if hasattr(past_key_values, m):
            try:
                out = getattr(past_key_values, m)()
                if out is not None:
                    return tuple(out)
            except Exception:
                pass

    if hasattr(past_key_values, "key_cache") and hasattr(past_key_values, "value_cache"):
        try:
            return tuple((k, v) for k, v in zip(past_key_values.key_cache, past_key_values.value_cache))
        except Exception:
            pass

    if hasattr(past_key_values, "layers"):
        pairs = []
        for layer in past_key_values.layers:
            k = getattr(layer, "keys", None)
            v = getattr(layer, "values", None)
            if k is None: k = getattr(layer, "key_cache", None)
            if v is None: v = getattr(layer, "value_cache", None)
            if k is None or v is None:
                raise TypeError(f"Unsupported layer cache object: {type(layer)}")
            pairs.append((k, v))
        return tuple(pairs)

    raise TypeError(f"Unsupported cache type: {type(past_key_values)}")


# def legacy_to_model_cache(legacy_cache, cache_cls=None):
#     if legacy_cache is None:
#         return None
#     if cache_cls is not None:
#         for m in ["from_legacy_cache", "from_legacy"]:
#             if hasattr(cache_cls, m):
#                 try:
#                     return getattr(cache_cls, m)(legacy_cache)
#                 except Exception:
#                     pass
#     # ✅ Build DynamicCache manually via .update() — stable in Transformers 5.x
#     try:
#         from transformers.cache_utils import DynamicCache
#         cache = DynamicCache()
#         for layer_idx, (k, v) in enumerate(legacy_cache):
#             cache.update(k, v, layer_idx)
#         return cache
#     except Exception:
#         return legacy_cache  # last resort

def legacy_to_model_cache(legacy_cache, cache_cls=None):
    if legacy_cache is None:
        return None
    from transformers.cache_utils import DynamicCache
    cache = DynamicCache()
    for layer_idx, (k, v) in enumerate(legacy_cache):
        cache.update(k, v, layer_idx)
    return cache
    
def _is_tensor_kv(k, v):
    return torch.is_tensor(k) and torch.is_tensor(v)


def _clone_kv_pair(k, v):
    k2 = k.detach() if torch.is_tensor(k) else k
    v2 = v.detach() if torch.is_tensor(v) else v
    return k2, v2


def extract_last_token_attn(attentions):
    if attentions is None:
        return None
    out = []
    for a in attentions:
        if torch.is_tensor(a):
            out.append(a[:, :, -1, :].detach())
        else:
            out.append(None)
    return out
from transformers import DynamicCache

def _ensure_dynamic_cache(past_kv):
    """Convert legacy tuple KV cache to DynamicCache if needed."""
    if past_kv is None:
        return None
    if isinstance(past_kv, tuple):
        return DynamicCache.from_legacy_cache(past_kv)
    return past_kv  # already a DynamicCache

In [ ]:
class KVCompressor:
    name = "base"

    def reset(self):
        pass

    def initialize(self, past_key_values, layer_attn=None):
        """Initialize internal compressed state from model output."""
        raise NotImplementedError

    def update(self, past_key_values, layer_attn=None):
        """Update internal state after one new decode step."""
        raise NotImplementedError

    def reconstruct_for_model(self):
        """Return standard past_key_values tuple for next model forward."""
        raise NotImplementedError

    def cache_bytes(self):
        """Approximate bytes used by internal cache representation."""
        return 0

In [9]:
class FullKV(KVCompressor):
    name = "fullkv"

    def reset(self):
        self.past = None

    def initialize(self, past_key_values, layer_attn=None):
        self.past = tuple(_clone_kv_pair(k, v) for k, v in past_key_values)

    def update(self, past_key_values, layer_attn=None):
        self.past = tuple(_clone_kv_pair(k, v) for k, v in past_key_values)

    def reconstruct_for_model(self):
        return self.past

    def cache_bytes(self):
        if self.past is None:
            return 0
        return sum(_bytes_tensor(k) + _bytes_tensor(v) for k, v in self.past)

In [8]:
class KVCompressor:
    name = "base"

    def reset(self):
        pass

    def initialize(self, past_key_values, layer_attn=None):
        """Initialize internal compressed state from model output."""
        raise NotImplementedError

    def update(self, past_key_values, layer_attn=None):
        """Update internal state after one new decode step."""
        raise NotImplementedError

    def reconstruct_for_model(self):
        """Return standard past_key_values tuple for next model forward."""
        raise NotImplementedError

    def cache_bytes(self):
        """Approximate bytes used by internal cache representation."""
        return 0

In [10]:
class ThinKV(KVCompressor):
    name = "thinkv"

    def __init__(self, max_budget_tokens=2048, sink_tokens=32, local_tokens=512, ema_alpha=0.9, thought_block=32, mode="compact"):
        self.max_budget_tokens = max_budget_tokens
        self.sink_tokens = sink_tokens
        self.local_tokens = local_tokens
        self.ema_alpha = ema_alpha
        self.thought_block = thought_block
        self.mode = mode  # "compact" (baseline) | "paged_sim"
        self.reset()

    def reset(self):
        self.past = None
        self.importance = None  # per-layer [B,H,T]
        self._paged_layers = []
        self._debug = {"evictions": 0, "recycled_blocks": 0, "active_blocks": 0}

    def _update_importance(self, layer_attn):
        if layer_attn is None:
            return
        if self.importance is None:
            self.importance = [a.detach().clone() if torch.is_tensor(a) else None for a in layer_attn]
            return
        new_imp = []
        for old, a in zip(self.importance, layer_attn):
            if not torch.is_tensor(a):
                new_imp.append(old if old is None else old)
                continue
            if old is None:
                new_imp.append(a.detach().clone())
                continue
            Tn = a.shape[-1]
            To = old.shape[-1]
            if Tn > To:
                pad = torch.zeros(*old.shape[:-1], Tn - To, device=old.device, dtype=old.dtype)
                old = torch.cat([old, pad], dim=-1)
            old = old[..., :Tn]
            merged = self.ema_alpha * old + (1 - self.ema_alpha) * a
            new_imp.append(merged)
        self.importance = new_imp

    def _evict_indices(self, T, layer_imp, device):
        if T <= self.max_budget_tokens:
            return torch.arange(T, device=device)

        sink = torch.arange(0, min(self.sink_tokens, T), device=device)
        local_start = max(self.sink_tokens, T - self.local_tokens)
        local = torch.arange(local_start, T, device=device)

        protected = torch.cat([sink, local]).unique(sorted=True)
        available = max(0, self.max_budget_tokens - protected.numel())
        if available <= 0:
            return protected

        middle_start = sink.numel()
        middle_end = local_start
        if middle_end <= middle_start:
            return protected

        cand = torch.arange(middle_start, middle_end, device=device)
        # thought groups by fixed block (proxy for CoT segments)
        groups = (cand - middle_start) // self.thought_block
        imp = layer_imp.mean(dim=(0, 1)).index_select(0, cand)

        # aggregate per thought group
        unique_g = torch.unique(groups)
        group_scores = []
        for g in unique_g:
            mask = (groups == g)
            score = imp[mask].sum()
            group_scores.append((score, g))

        # keep highest-score groups until budget filled
        group_scores.sort(key=lambda x: float(x[0]), reverse=True)
        keep_tokens = []
        left = available
        for _, g in group_scores:
            tok = cand[groups == g]
            if tok.numel() <= left:
                keep_tokens.append(tok)
                left -= tok.numel()
            else:
                if left > 0:
                    keep_tokens.append(tok[:left])
                    left = 0
            if left <= 0:
                break

        kept_mid = torch.cat(keep_tokens) if keep_tokens else torch.empty(0, dtype=torch.long, device=device)
        return torch.cat([protected, kept_mid]).unique(sorted=True)

    def _empty_paged_layer(self):
        return {
            "k_slots": [],
            "v_slots": [],
            "logical_to_phys": {},
            "logical_to_len": {},
            "logical_order": [],
            "free_list": [],
        }

    def _compress_layer_compact(self, k, v, imp):
        idx = self._evict_indices(k.shape[2], imp, k.device)
        return k.index_select(2, idx), v.index_select(2, idx), imp.index_select(-1, idx), idx

    def _compress_layer_paged(self, k, v, imp, layer_state):
        idx = self._evict_indices(k.shape[2], imp, k.device)
        idx_list = idx.tolist()
        old_map = dict(layer_state["logical_to_phys"])

        block = max(1, int(self.thought_block))
        keep_by_logical = {}
        for pos in idx_list:
            logical_id = pos // block
            keep_by_logical.setdefault(logical_id, []).append(pos)

        new_logical = sorted(keep_by_logical.keys())

        # invalidate old logical mappings first; reclaimed slots go to free-list
        removed = set(old_map.keys()) - set(new_logical)
        recycled = 0
        for lid in removed:
            phys = old_map[lid]
            layer_state["logical_to_phys"].pop(lid, None)
            layer_state["logical_to_len"].pop(lid, None)
            layer_state["free_list"].append(phys)
            recycled += 1

        for lid in new_logical:
            phys = layer_state["logical_to_phys"].get(lid)
            if phys is None:
                if layer_state["free_list"]:
                    phys = layer_state["free_list"].pop()
                    recycled += 1
                else:
                    phys = len(layer_state["k_slots"])
                    layer_state["k_slots"].append(None)
                    layer_state["v_slots"].append(None)
                layer_state["logical_to_phys"][lid] = phys

            pos = torch.tensor(keep_by_logical[lid], device=k.device, dtype=torch.long)
            kb = k.index_select(2, pos).detach()
            vb = v.index_select(2, pos).detach()
            layer_state["k_slots"][phys] = kb
            layer_state["v_slots"][phys] = vb
            layer_state["logical_to_len"][lid] = int(pos.numel())

        layer_state["logical_order"] = new_logical

        self._debug["evictions"] += max(0, int(k.shape[2] - idx.numel()))
        self._debug["recycled_blocks"] += recycled

        return imp.index_select(-1, idx)

    def initialize(self, past_key_values, layer_attn=None):
        self.past = tuple(_clone_kv_pair(k, v) for k, v in past_key_values)
        self._update_importance(layer_attn)
        self._compress_now()

    def update(self, past_key_values, layer_attn=None):
        self.past = tuple(_clone_kv_pair(k, v) for k, v in past_key_values)
        self._update_importance(layer_attn)
        self._compress_now()

    def _compress_now(self):
        if self.past is None:
            return

        if self.mode == "compact":
            new_layers = []
            new_imp = []
            for li, (k, v) in enumerate(self.past):
                if not _is_tensor_kv(k, v):
                    new_layers.append((k, v))
                    if self.importance is not None:
                        new_imp.append(self.importance[li] if li < len(self.importance) else None)
                    continue
                T = k.shape[2]
                imp = (
                    self.importance[li]
                    if (self.importance is not None and li < len(self.importance) and torch.is_tensor(self.importance[li]))
                    else torch.ones(k.shape[0], k.shape[1], T, device=k.device)
                ).to(k.device)
                ck, cv, cimp, _ = self._compress_layer_compact(k, v, imp)
                new_layers.append((ck, cv))
                if self.importance is not None:
                    new_imp.append(cimp)
            self.past = tuple(new_layers)
            if self.importance is not None:
                self.importance = new_imp
            self._paged_layers = []
            self._debug["active_blocks"] = 0
            return

        if self.mode != "paged_sim":
            raise ValueError(f"Unknown ThinKV mode: {self.mode}")

        if len(self._paged_layers) != len(self.past):
            self._paged_layers = [self._empty_paged_layer() for _ in self.past]

        new_imp = []
        active = 0
        for li, (k, v) in enumerate(self.past):
            if not _is_tensor_kv(k, v):
                if self.importance is not None:
                    new_imp.append(self.importance[li] if li < len(self.importance) else None)
                continue

            T = k.shape[2]
            imp = (
                self.importance[li]
                if (self.importance is not None and li < len(self.importance) and torch.is_tensor(self.importance[li]))
                else torch.ones(k.shape[0], k.shape[1], T, device=k.device)
            ).to(k.device)

            cimp = self._compress_layer_paged(k, v, imp, self._paged_layers[li])
            if self.importance is not None:
                new_imp.append(cimp)
            active += len(self._paged_layers[li]["logical_to_phys"])

        if self.importance is not None:
            self.importance = new_imp
        self._debug["active_blocks"] = active

    def reconstruct_for_model(self):
        if self.mode != "paged_sim":
            return self.past

        if self.past is None:
            return None

        rebuilt = []
        for li, (k, v) in enumerate(self.past):
            if not _is_tensor_kv(k, v):
                rebuilt.append((k, v))
                continue

            layer_state = self._paged_layers[li]
            chunks_k, chunks_v = [], []
            for lid in layer_state["logical_order"]:
                phys = layer_state["logical_to_phys"][lid]
                kb = layer_state["k_slots"][phys]
                vb = layer_state["v_slots"][phys]
                if kb is None or vb is None:
                    continue
                chunks_k.append(kb)
                chunks_v.append(vb)

            if chunks_k:
                rebuilt.append((torch.cat(chunks_k, dim=2), torch.cat(chunks_v, dim=2)))
            else:
                rebuilt.append((k[:, :, :0, :], v[:, :, :0, :]))

        return tuple(rebuilt)

    def debug_counters(self):
        return dict(self._debug)

    def cache_bytes(self):
        if self.past is None:
            return 0

        if self.mode != "paged_sim":
            val = sum(_bytes_tensor(k) + _bytes_tensor(v) for k, v in self.past)
            if self.importance is not None:
                val += sum(_bytes_tensor(x) for x in self.importance)
            return val

        val = 0
        for layer_state in self._paged_layers:
            for kb, vb in zip(layer_state["k_slots"], layer_state["v_slots"]):
                if torch.is_tensor(kb):
                    val += _bytes_tensor(kb)
                if torch.is_tensor(vb):
                    val += _bytes_tensor(vb)
        if self.importance is not None:
            val += sum(_bytes_tensor(x) for x in self.importance)
        return val



In [11]:
class MiniCache(KVCompressor):
    name = "minicache"

    def __init__(self, angle_threshold=0.95):
        self.angle_threshold = angle_threshold
        self.reset()

    def reset(self):
        self.n_layers = 0
        self.pairs = []
        self.shared = {}  # (l,l1)-> dict with shared dir/mag/mask/original_unmerged

    def _compress_pair(self, K0, V0, K1, V1):
        # K/V: [B,H,T,D]
        dir0, mag0 = _norm_dir(K0)
        dir1, mag1 = _norm_dir(K1)
        cos = (dir0 * dir1).sum(dim=-1, keepdim=True)
        shared_dir = F.normalize(dir0 + dir1, dim=-1)
        mask = (cos > self.angle_threshold)

        final_dir = torch.where(mask, shared_dir, dir0)
        data = {
            "k_dir": final_dir.detach(),
            "k_mag0": mag0.detach(),
            "k_mag1": mag1.detach(),
            "k_mask": mask.detach(),
            "k_unmerged1": K1.detach(),
            "v0": V0.detach(),
            "v1": V1.detach(),
        }
        return data

    def _reconstruct_pair(self, data):
        K0 = data["k_dir"] * data["k_mag0"]
        K1_merge = data["k_dir"] * data["k_mag1"]
        K1 = torch.where(data["k_mask"], K1_merge, data["k_unmerged1"])
        return K0, data["v0"], K1, data["v1"]

    def initialize(self, past_key_values, layer_attn=None):
        self.n_layers = len(past_key_values)
        self.pairs = _pair_layers(self.n_layers)
        self.shared = {}
        for l0, l1 in self.pairs:
            if l1 is None:
                self.shared[(l0, l1)] = {"single": (past_key_values[l0][0].detach(), past_key_values[l0][1].detach())}
            else:
                K0, V0 = past_key_values[l0]
                K1, V1 = past_key_values[l1]
                if _is_tensor_kv(K0, V0) and _is_tensor_kv(K1, V1):
                    self.shared[(l0, l1)] = self._compress_pair(K0, V0, K1, V1)
                else:
                    self.shared[(l0, l1)] = {"fallback": (_clone_kv_pair(K0, V0), _clone_kv_pair(K1, V1))}

    def update(self, past_key_values, layer_attn=None):
        self.initialize(past_key_values, layer_attn)

    def reconstruct_for_model(self):
        layers = [None] * self.n_layers
        for (l0, l1), data in self.shared.items():
            if l1 is None:
                layers[l0] = data["single"]
            else:
                if "fallback" in data:
                    layers[l0] = data["fallback"][0]
                    layers[l1] = data["fallback"][1]
                else:
                    K0, V0, K1, V1 = self._reconstruct_pair(data)
                    layers[l0] = (K0, V0)
                    layers[l1] = (K1, V1)
        return _stack_layers(layers)

    def cache_bytes(self):
        total = 0
        for (_, _), d in self.shared.items():
            for v in d.values():
                if torch.is_tensor(v):
                    total += _bytes_tensor(v)
                elif isinstance(v, tuple):
                    total += sum(_bytes_tensor(x) for x in v)
        return total

In [12]:
class CommonKV(KVCompressor):
    name = "commonkv"

    def __init__(self, base_shared_ratio=0.75, min_shared_ratio=0.40, max_shared_ratio=0.95):
        self.base_shared_ratio = base_shared_ratio
        self.min_shared_ratio = min_shared_ratio
        self.max_shared_ratio = max_shared_ratio
        self.reset()

    def reset(self):
        self.n_layers = 0
        self.pairs = []
        self.data = {}

    def _adaptive_ratio(self, X0, X1):
        sim = F.cosine_similarity(X0.flatten(2), X1.flatten(2), dim=-1).mean()
        sim_f = float(torch.clamp(sim, -1, 1))
        ratio = self.base_shared_ratio + 0.2 * sim_f
        ratio = max(self.min_shared_ratio, min(self.max_shared_ratio, ratio))
        return ratio

    def _compress_pair(self, K0, V0, K1, V1):
        # shared latent + residuals
        ratio = self._adaptive_ratio(K0, K1)
        shared_k = 0.5 * (K0 + K1)
        shared_v = 0.5 * (V0 + V1)
        dK0 = K0 - shared_k
        dK1 = K1 - shared_k
        dV0 = V0 - shared_v
        dV1 = V1 - shared_v

        # adaptive residual sparsification budget
        keep = max(1, int(shared_k.shape[2] * ratio))
        if keep < shared_k.shape[2]:
            idx = torch.linspace(0, shared_k.shape[2]-1, keep, device=shared_k.device).long()
            shared_k = shared_k.index_select(2, idx)
            shared_v = shared_v.index_select(2, idx)
            dK0 = dK0.index_select(2, idx)
            dK1 = dK1.index_select(2, idx)
            dV0 = dV0.index_select(2, idx)
            dV1 = dV1.index_select(2, idx)

        return {
            "shared_k": shared_k.detach(),
            "shared_v": shared_v.detach(),
            "dK0": dK0.detach(),
            "dK1": dK1.detach(),
            "dV0": dV0.detach(),
            "dV1": dV1.detach(),
        }

    def initialize(self, past_key_values, layer_attn=None):
        self.n_layers = len(past_key_values)
        self.pairs = _pair_layers(self.n_layers)
        self.data = {}
        for l0, l1 in self.pairs:
            if l1 is None:
                self.data[(l0, l1)] = {"single": (past_key_values[l0][0].detach(), past_key_values[l0][1].detach())}
            else:
                K0, V0 = past_key_values[l0]
                K1, V1 = past_key_values[l1]
                if _is_tensor_kv(K0, V0) and _is_tensor_kv(K1, V1):
                    self.data[(l0, l1)] = self._compress_pair(K0, V0, K1, V1)
                else:
                    self.data[(l0, l1)] = {"fallback": (_clone_kv_pair(K0, V0), _clone_kv_pair(K1, V1))}

    def update(self, past_key_values, layer_attn=None):
        self.initialize(past_key_values, layer_attn)

    def reconstruct_for_model(self):
        layers = [None] * self.n_layers
        for (l0, l1), d in self.data.items():
            if l1 is None:
                layers[l0] = d["single"]
            else:
                if "fallback" in d:
                    layers[l0] = d["fallback"][0]
                    layers[l1] = d["fallback"][1]
                else:
                    K0 = d["shared_k"] + d["dK0"]
                    K1 = d["shared_k"] + d["dK1"]
                    V0 = d["shared_v"] + d["dV0"]
                    V1 = d["shared_v"] + d["dV1"]
                    layers[l0] = (K0, V0)
                    layers[l1] = (K1, V1)
        return _stack_layers(layers)

    def cache_bytes(self):
        total = 0
        for (_, _), d in self.data.items():
            for v in d.values():
                if torch.is_tensor(v):
                    total += _bytes_tensor(v)
                elif isinstance(v, tuple):
                    total += sum(_bytes_tensor(x) for x in v)
        return total

In [13]:
class PaLu(KVCompressor):
    name = "palu"

    def __init__(self, rank=64):
        self.rank = rank
        self.reset()

    def reset(self):
        self.lowrank = []  # per layer: tuples of z_k,z_v,b_k,b_v
        self.n_layers = 0

    def _factorize(self, X, r):
        # X: [B,H,T,D], compress hidden dim D -> r, keep token dimension T
        B, H, T, D = X.shape
        X2 = X.reshape(B*H*T, D)
        m = X2.mean(0, keepdim=True)
        Xc = X2 - m
        try:
            U, S, Vh = torch.linalg.svd(Xc, full_matrices=False)
            rr = min(r, Vh.shape[0])
            Bm = Vh[:rr, :]            # [r,D]
            Z = X2 @ Bm.t()            # [BHT,r]
            return Z.reshape(B, H, T, rr), Bm, m
        except Exception:
            # fallback no compression
            eye = torch.eye(D, device=X.device, dtype=X.dtype)
            return X, eye, torch.zeros_like(m)

    def _reconstruct(self, Z, Bm, mean, D):
        if Bm.shape[0] == D and torch.allclose(Bm, torch.eye(D, device=Bm.device, dtype=Bm.dtype)):
            return Z
        B, H, T, R = Z.shape
        Z2 = Z.reshape(B*H*T, R)
        X2 = Z2 @ Bm + mean
        return X2.reshape(B, H, T, D)

    def initialize(self, past_key_values, layer_attn=None):
        self.n_layers = len(past_key_values)
        self.lowrank = []
        for k, v in past_key_values:
            if not _is_tensor_kv(k, v):
                self.lowrank.append((k, v, None, None, None, None, None, None))
                continue
            Dk = k.shape[-1]
            Dv = v.shape[-1]
            zk, bk, mk = self._factorize(k.detach(), self.rank)
            zv, bv, mv = self._factorize(v.detach(), self.rank)
            self.lowrank.append((zk, zv, bk, bv, mk, mv, Dk, Dv))

    def update(self, past_key_values, layer_attn=None):
        self.initialize(past_key_values, layer_attn)

    def reconstruct_for_model(self):
        layers = []
        for (zk, zv, bk, bv, mk, mv, Dk, Dv) in self.lowrank:
            if bk is None:
                layers.append((zk, zv))
                continue
            k = self._reconstruct(zk, bk, mk, Dk)
            v = self._reconstruct(zv, bv, mv, Dv)
            layers.append((k, v))
        return _stack_layers(layers)

    def cache_bytes(self):
        total = 0
        for (zk, zv, bk, bv, mk, mv, _, _) in self.lowrank:
            for t in [zk, zv, bk, bv, mk, mv]:
                if torch.is_tensor(t):
                    total += _bytes_tensor(t)
        return total

In [14]:
@torch.no_grad()
def generate_with_compressor(prompt: str, compressor: KVCompressor, max_new_tokens=128, temperature=0.0):
    compressor.reset()
    inp = tokenizer(prompt, return_tensors="pt")
    inp = {k: v.to(model.device) for k, v in inp.items()}

    out = model(**inp, use_cache=True, output_attentions=True)
    layer_attn = extract_last_token_attn(out.attentions if hasattr(out, "attentions") else None)
    cache_cls = type(out.past_key_values)
    compressor.initialize(cache_to_legacy(out.past_key_values), layer_attn=layer_attn)

    logits = out.logits[:, -1, :]
    gen_ids = []

    for step in range(max_new_tokens):
        if temperature > 0:
            probs = torch.softmax(logits / max(temperature, 1e-6), dim=-1)
            next_tok = torch.multinomial(probs, num_samples=1)
        else:
            # greedy with eos-avoid at first step to prevent empty generations
            top2 = torch.topk(logits, k=2, dim=-1).indices
            next_tok = top2[:, :1]
            if step == 0 and tokenizer.eos_token_id is not None and int(next_tok.item()) == int(tokenizer.eos_token_id):
                next_tok = top2[:, 1:2]

        tok = int(next_tok.item())
        if tokenizer.eos_token_id is not None and tok == int(tokenizer.eos_token_id):
            break
        gen_ids.append(tok)

        past_for_model = legacy_to_model_cache(compressor.reconstruct_for_model(), cache_cls=cache_cls)
        out = model(
            input_ids=next_tok,
            past_key_values=past_for_model,
            use_cache=True,
            output_attentions=True,
        )
        layer_attn = extract_last_token_attn(out.attentions if hasattr(out, "attentions") else None)
        compressor.update(cache_to_legacy(out.past_key_values), layer_attn=layer_attn)
        logits = out.logits[:, -1, :]

    text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
    return text, compressor.cache_bytes()


In [17]:
TOKEN_RE = re.compile(r"\w+", re.UNICODE)

def _words(s):
    return TOKEN_RE.findall((s or "").lower())

def em(pred, refs):
    p = (pred or "").strip().lower()
    return float(any(p == (r or "").strip().lower() for r in refs))

def f1(pred, refs):
    p = _words(pred)
    if not p: return 0.0
    best=0.0
    for r in refs:
        t=_words(r)
        if not t: continue
        c=0
        cnt={}
        for x in t: cnt[x]=cnt.get(x,0)+1
        for x in p:
            if cnt.get(x,0)>0:
                c+=1; cnt[x]-=1
        if c==0: continue
        pr=c/len(p); rc=c/len(t)
        best=max(best,2*pr*rc/(pr+rc))
    return best

def score(pred, refs, metric):
    m=metric.lower()
    if m in {"em","exact_match"}: return em(pred, refs)
    return f1(pred, refs)

@dataclass
class Ex:
    eid: str
    benchmark: str
    group: str
    task: str
    prompt: str
    refs: List[str]
    metric: str


def load_longbench(task_specs, split="test", limit=20):
    from datasets import load_dataset
    rows = []
    base_dir = Path("./longbench_data/data")  # adjust to where you unzipped

    for s in task_specs:
        task = s["task"]
        file_path = base_dir / f"{task}.jsonl"
        # ds = load_dataset("THUDM/LongBench", task, split=split)
        # ds = load_dataset("THUDM/LongBench", "narrative", split="test")
        # Read JSONL into list of dicts
        records = []
        with open(file_path, "r", encoding="utf-8") as f:
            for line in f:
                records.append(json.loads(line))

        # Wrap into a Dataset object
        ds = Dataset.from_list(records)
        if limit: ds=ds.select(range(min(limit,len(ds))))
        for i,ex in enumerate(ds):
            prompt=ex.get("prompt") or ex.get("input") or ""
            if not prompt and "context" in ex and "question" in ex:
                prompt="Context:\n{}\n\nQuestion: {}\nAnswer:".format(ex['context'], ex['question'])
            refs=ex.get("answers") or ex.get("answer") or ex.get("output") or [""]
            if not isinstance(refs,list): refs=[str(refs)]
            rows.append(Ex(f"lb::{s['task']}::{i}","LongBench",s["group"],s["task"],str(prompt),[str(x) for x in refs],s.get("metric","f1")))
    return rows


def load_ruler(root_dir, task_specs, limit=20):
    rows=[]
    root=Path(root_dir)
    for s in task_specs:
        fp=root/f"{s['task']}.jsonl"
        if not fp.exists():
            print("missing", fp)
            continue
        with fp.open("r", encoding="utf-8") as f:
            for i,line in enumerate(f):
                if limit and i>=limit: break
                ex=json.loads(line)
                prompt=ex.get("prompt") or ex.get("input") or ""
                refs=ex.get("answers") or ex.get("answer") or ex.get("output") or [""]
                if not isinstance(refs,list): refs=[str(refs)]
                rows.append(Ex(f"ru::{s['task']}::{i}","Ruler",s["group"],s["task"],str(prompt),[str(x) for x in refs],s.get("metric","em")))
    return rows

In [ ]:
COMPRESSORS = {
    "fullkv": FullKV(),
    "thinkv": ThinKV(max_budget_tokens=2048, sink_tokens=32, local_tokens=512, mode="compact"),
    "thinkv_paged": ThinKV(max_budget_tokens=2048, sink_tokens=32, local_tokens=512, mode="paged_sim"),
    "minicache": MiniCache(angle_threshold=0.95),
    "commonkv": CommonKV(base_shared_ratio=0.75),
    "palu": PaLu(rank=64),
}

LONGBENCH_TASKS = [
    {"task":"narrativeqa","group":"QA","metric":"f1"},
    {"task":"gov_report","group":"Sum.","metric":"f1"},
    {"task":"trec","group":"FShot","metric":"em"},
    {"task":"hotpotqa","group":"Synth.","metric":"f1"},
    {"task":"lcc","group":"Code","metric":"em"},
]
RULER_TASKS = [
    # group labels: NS, NMK, NMQ, NMV, QA, VT, FWE, CWE
    {"task": "cwe", "group": "CWE", "metric": "em"},
    {"task": "niah_multikey_1", "group": "NMK", "metric": "em"},
    {"task": "niah_multikey_2", "group": "NMK", "metric": "em"},
    {"task": "niah_multikey_3", "group": "NMK", "metric": "em"},
    {"task": "niah_multiquery", "group": "NMQ", "metric": "em"},
    {"task": "niah_multivalue", "group": "NMV", "metric": "em"},
    {"task": "niah_single_1", "group": "NS", "metric": "em"},
    {"task": "niah_single_2", "group": "NS", "metric": "em"},
    {"task": "niah_single_3", "group": "NS", "metric": "em"},
    {"task": "qa_1", "group": "QA", "metric": "f1"},
    {"task": "qa_2", "group": "QA", "metric": "f1"},
    {"task": "vt", "group": "VT", "metric": "em"},
    {"task": "fwe", "group": "FWE", "metric": "em"},
]

examples=[]
examples += load_longbench(LONGBENCH_TASKS, split="test", limit=10)
examples += load_ruler("./ruler_data", RULER_TASKS, limit=10)
print("examples", len(examples))

rows=[]
for ex in tqdm(examples, desc="eval"):
    for name, comp in COMPRESSORS.items():
        t0=time.perf_counter()
        err=""
        pred=""
        cbytes=0
        try:
            pred, cbytes = generate_with_compressor(ex.prompt, comp, max_new_tokens=128, temperature=0.0)
            if pred == "":
                err = "empty_generation"
        except Exception as e:
            err=str(e)
        dt=time.perf_counter()-t0

        sc=0.0 if err else score(pred, ex.refs, ex.metric)
        dbg = comp.debug_counters() if hasattr(comp, "debug_counters") else {}
        rows.append({
            "eid":ex.eid,"benchmark":ex.benchmark,"group":ex.group,"task":ex.task,
            "method":name,"metric":ex.metric,"score":sc,"latency_s":dt,
            "cache_bytes":cbytes,"error":err,"pred":pred,
            "debug_evictions": dbg.get("evictions", 0),
            "debug_recycled_blocks": dbg.get("recycled_blocks", 0),
            "debug_active_blocks": dbg.get("active_blocks", 0),
        })

results_df = pd.DataFrame(rows)
results_df["is_error"] = results_df["error"].astype(str).str.len() > 0

summary = results_df.pivot_table(index="method", columns=["benchmark","group"], values="score", aggfunc="mean")
summary.columns=[f"{b}:{g}" for b,g in summary.columns]
summary["Avg."] = results_df.groupby("method")["score"].mean()
summary["Latency(s)"] = results_df.groupby("method")["latency_s"].mean()
summary["CacheMB"] = results_df.groupby("method")["cache_bytes"].mean()/(1024**2)
summary["ErrorRate"] = results_df.groupby("method")["is_error"].mean()
summary = summary.sort_values("Avg.", ascending=False)

print("Any errors?", results_df["is_error"].any())
if results_df["is_error"].any():
    display(results_df[results_df["is_error"]][["method","task","error"]].head(20))

summary



examples 180


eval:   0%|          | 0/180 [00:00<?, ?it/s]

In [ ]:
out=Path("artifacts")
out.mkdir(exist_ok=True)
results_df.to_csv(out/"kvcache_eval_raw.csv", index=False)
summary.to_csv(out/"kvcache_eval_summary.csv")
print("saved artifacts/kvcache_eval_raw.csv and artifacts/kvcache_eval_summary.csv")